In [1]:
%pip install llama-index-llms-together

Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip install llama-index

In [3]:
import os
from llama_index.llms.together import TogetherLLM
from llama_index.core.llms import ChatMessage

os.environ["TOGETHER_API_KEY"] = '6379c9bdda7f5f5cbb14abbce2046110e50b0cc7aee25eb809da8bc6fa2a43d0'

/home/thanhtruongtran/Desktop/RecSys/post-generation/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
cd ..


/home/thanhtruongtran/Desktop/RecSys/post-generation


/home/thanhtruongtran/Desktop/RecSys/post-generation/venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [42]:
from models.embedding_models import EmbeddingModel
from llama_index.core.vector_stores import VectorStoreQuery
from databases.postgresql import PostgresDB
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
# initialize
embed_model = EmbeddingModel()._embed_model
vector_store=PostgresDB().vector_store

#test
query_str = "aave, gpt"
query_embedding = embed_model.get_query_embedding(query_str)
query_mode = "default"
vector_store_query = VectorStoreQuery(
    query_embedding=query_embedding, similarity_top_k=20, mode=query_mode
)

query_result = vector_store.query(vector_store_query)
tweet_simi =dict()
for i in range(0, 20):
    tweet_simi[f'post {i}'] = query_result.nodes[i].get_content()

In [50]:
import random
llm = TogetherLLM(
    model='mistralai/mixtral-8x7b-instruct-v0.1', api_key='6379c9bdda7f5f5cbb14abbce2046110e50b0cc7aee25eb809da8bc6fa2a43d0',
    temperature=0.75, max_length=500, top_p= 1
)

prompt = [
    ChatMessage(
        role="user", content="""
            You are a professional marketer and your task is to create a post for a social media platform.
            Now, I will give you my requirement and after I say 'Start!!!', you will start your task.
        """
    ),
    ChatMessage(
        role="system", content="Yeah, I can help you with that. What is your requirement?"
    ),
    ChatMessage(
        role="user", content="""
            I want to create a post for a social media platform with the keywords: {query_str};
            You must write a post suitable for a social media platform.
            - If I want to create a post for Twitter, please limit your post to maximum 280 characters.
            - If I want to create a post for Telegram or Discord, consider your post with longer content and more detailed information.
            I will provide you with some examples of posts. Please wait a moment.
        """
    ),
    ChatMessage(
        role="system", content="Sure, I can help you with that. Please provide me with the social media platform you want to create a post for, and examples of posts."
    ),
    ChatMessage(
        role="user", content=f"""
            I want to create a post for Twitter. 
            Here are some examples of posts:
                - Post1 content: {tweet_simi[f'post {0}']}
                - Post2 content: {tweet_simi[f'post {1}']}
                - Post3 content: {tweet_simi[f'post {2}']}
                - Post4 content: {tweet_simi[f'post {3}']}
                - Post5 content: {tweet_simi[f'post {4}']}
            Please write a post in the same style and layout as the above posts. Your post should only contain content without any unnecessary information such as "Author", "Note", "Introduction", or "Conclusion".
        """
    ),
    ChatMessage(
        role="system", content="Great! I have understood your requirement. Do you have any other requirements?"
    ),
    ChatMessage(
        role="user", content="""
            Remember that:
                - Your post should only contain content, no additional information or introduction or conclusion.
                - Limit the number of "\\n" in your post to a maximum of 2. Use line breaks only when you want to write a new part of your post.
        """
    ),
    ChatMessage(
        role="system", content="Okay, I have understood your requirement. Let's start your task now."
    ),
    ChatMessage(
        role="user", content="Start!!!"
    ),
]

result = llm.stream_chat(prompt)


In [51]:
for r in result:
    print(r.delta, end="")

 "🚀 Exciting news from Aave Labs! Proposal for cross-chain GHO rollout now live, inviting networks to propose potential destinations for GHO. Join the discussion and help shape the future of Aave's decentralized money market: <https://t.co/49VkBNT0nB> #Aave #DeFi #CrossChain"

In [55]:
print(" Exciting news for the Oraichain community! 🚀 The Oraichain Developer Advocate Program is now live, and it's a game changer! 🖥️\n\nThis program is not just for developers, but for anyone who believes in Orai's vision. It's structured in 3 parts:\n\n1️⃣ Developer Evangelist: Evangelize, explore documentation, and earn Experience Points.\n\n2️⃣ Developer Advocate: Build projects, onboard evangelists, and represent the community.\n\n3️⃣ Core Committee: Become a Grant Winner, start a validator node, and actively contribute.\n\nJoining the program means:\n\n* Being part of a fast-growing ecosystem 📈\n* Diving into cutting-edge technology 🤖\n* Professional growth and networking opportunities 🌐\n* Mentorship and support from a network of mentors 👨‍🏫\n* Building your portfolio 💼\n* Incentives and rewards 💰\n\nDon't miss this chance to shape the future of blockchain and AI! 🤝\n\n👉 More info: <https://t.co/VSMJ8aITKk>\n\n#oraichain #blockchain #ai #developeradvocate #program #opportunity #incentives #rewards #community")

 Exciting news for the Oraichain community! 🚀 The Oraichain Developer Advocate Program is now live, and it's a game changer! 🖥️

This program is not just for developers, but for anyone who believes in Orai's vision. It's structured in 3 parts:

1️⃣ Developer Evangelist: Evangelize, explore documentation, and earn Experience Points.

2️⃣ Developer Advocate: Build projects, onboard evangelists, and represent the community.

3️⃣ Core Committee: Become a Grant Winner, start a validator node, and actively contribute.

Joining the program means:

* Being part of a fast-growing ecosystem 📈
* Diving into cutting-edge technology 🤖
* Professional growth and networking opportunities 🌐
* Mentorship and support from a network of mentors 👨‍🏫
* Building your portfolio 💼
* Incentives and rewards 💰

Don't miss this chance to shape the future of blockchain and AI! 🤝

👉 More info: <https://t.co/VSMJ8aITKk>

#oraichain #blockchain #ai #developeradvocate #program #opportunity #incentives #rewards #communit

In [56]:
print(list(range(5, 100, 5)))

[5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95]
